In [1]:
import sys
import logging
from typing import Dict, Any, List, Optional, Annotated
from dataclasses import dataclass
from enum import Enum
from typing_extensions import TypedDict

# 模拟 LangGraph 核心组件导入（若环境中安装了 langgraph，可直接引入）
# from langgraph.graph import StateGraph, END
# from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

logging.basicConfig(level=logging.INFO, format="%(asctime)s - [%(levelname)s] - %(message)s")
logger = logging.getLogger("PolicyGateGraph")

# ============================================================================
# 1. 状态与决策计划定义 (State & Execution Plan)
# ============================================================================

class GateDecision(str, Enum):
    ALLOWED = "ALLOWED"           # 允许直接执行（如只读工具）
    PENDING_CONFIRMATION = "PENDING_CONFIRMATION"  # 需挂起等待确认（如转账）
    DENIED = "DENIED"             # 直接拒绝（如无权限或余额不足取消）

class AgentState(TypedDict):
    messages: List[Any]           # 对话与工具消息历史
    execution_plan: List[Dict[str, Any]] # Policy Gate 生成的执行计划
    balance: Optional[float]      # 当前账户余额缓存
    debit_count: int              # 实际扣款计数器（用于断言验证）
    approval_given: bool          # 是否获得人工确认授权

# ============================================================================
# 2. 模拟工具实现 (Tools)
# ============================================================================

class MockBackend:
    def __init__(self, initial_balance: float = 500.0):
        self.balance = initial_balance
        self.debit_count = 0

    def get_balance(self) -> dict:
        logger.info("⚙️ [Tool Execution] 执行只读工具: get_balance()")
        return {"balance": self.balance}

    def execute_transfer(self, amount: float, recipient: str) -> dict:
        logger.warning(f"🚨 [Tool Execution] 执行高危副作用工具: execute_transfer(amount={amount}, recipient='{recipient}')")
        if amount > self.balance:
            return {"success": False, "reason": "INSUFFICIENT_FUNDS"}
        self.balance -= amount
        self.debit_count += 1
        return {"success": True, "remaining_balance": self.balance}

backend = MockBackend(initial_balance=500.0) # 设初始余额为 500

# ============================================================================
# 3. LangGraph 节点定义 (Nodes)
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    """
    1. 模拟 LLM 决策节点：一次性生成多个 Tool Call
    包含一个只读工具 get_balance 和一个高危转账工具 execute_transfer (申请转 1000)
    """
    logger.info("🤖 [agent_node] LLM 正在生成批量 Tool Calls...")
    
    mock_tool_calls = [
        {"id": "call_001", "name": "get_balance", "args": {}},
        {"id": "call_002", "name": "execute_transfer", "args": {"amount": 1000.0, "recipient": "Alice"}}
    ]
    
    # 模拟 AIMessage
    ai_message = {"role": "assistant", "tool_calls": mock_tool_calls, "content": "我来帮您查询余额并完成转账。"}
    state["messages"].append(ai_message)
    return state


def policy_gate_node(state: AgentState) -> AgentState:
    """
    2. Policy Gate 独立评估节点：
    对每个 Tool Call 进行独立审查，生成 execution_plan。
    """
    logger.info("🛡️ [policy_gate_node] 正在对所有 Tool Calls 进行独立契约与安全审查...")
    
    latest_msg = state["messages"][-1]
    tool_calls = latest_msg.get("tool_calls", [])
    plan = []
    
    for tc in tool_calls:
        t_name = tc["name"]
        if t_name == "get_balance":
            # 只读工具：直接放行
            plan.append({"id": tc["id"], "name": t_name, "args": tc["args"], "decision": GateDecision.ALLOWED})
        elif t_name == "execute_transfer":
            # 副作用工具：默认挂起，绝不直接放行
            # 检查是否有显式授权与确认
            if not state.get("approval_given", False):
                plan.append({"id": tc["id"], "name": t_name, "args": tc["args"], "decision": GateDecision.PENDING_CONFIRMATION, "reason": "写操作缺少人工确认授权"})
            else:
                plan.append({"id": tc["id"], "name": t_name, "args": tc["args"], "decision": GateDecision.ALLOWED})
                
    state["execution_plan"] = plan
    return state


def safe_execution_node(state: AgentState) -> AgentState:
    """
    3. 安全执行节点：
    严格根据 Policy Gate 的执行计划行动。
    - 先执行 ALLOW 的只读工具 (如 get_balance)。
    - 获取结果后进行业务预检：如果余额不足，取消/拒绝待执行的转账。
    - 处于 PENDING_CONFIRMATION 的转账工具绝不执行。
    """
    logger.info("⚡ [safe_execution_node] 严格根据安全执行计划调度工具...")
    plan = state.get("execution_plan", [])
    
    for item in plan:
        if item["decision"] == GateDecision.ALLOWED and item["name"] == "get_balance":
            # 执行只读工具
            res = backend.get_balance()
            state["balance"] = res["balance"]
            state["messages"].append({"role": "tool", "tool_call_id": item["id"], "name": item["name"], "content": str(res)})
            
            # 【业务预检逻辑】如果余额不足，自动取消后续所有待执行的转账计划！
            if state["balance"] < 1000.0:
                logger.warning(f"⚠️ [Business Guard] 余额检查结果 ({state['balance']}) 小于转账金额！自动取消后续待执行转账。")
                for sub_item in plan:
                    if sub_item["name"] == "execute_transfer":
                        sub_item["decision"] = GateDecision.DENIED
                        state["messages"].append({
                            "role": "tool", 
                            "tool_call_id": sub_item["id"], 
                            "name": sub_item["name"], 
                            "content": "ERROR: CANCELLED_Due_To_Insufficient_Balance"
                        })
                        
        elif item["decision"] == GateDecision.ALLOWED and item["name"] == "execute_transfer":
            # 只有当且仅当获得授权且预检通过后才执行转账
            args = item["args"]
            res = backend.execute_transfer(amount=args["amount"], recipient=args["recipient"])
            state["messages"].append({"role": "tool", "tool_call_id": item["id"], "name": item["name"], "content": str(res)})
            
        elif item["decision"] == GateDecision.PENDING_CONFIRMATION:
            logger.info(f"🔒 [Gate Interception] 工具 [{item['name']}] 处于挂起状态，已成功拦截，未执行。")
            
    return state

# ============================================================================
# 4. 测试场景与严格断言验证 (Verification & Assertions)
# ============================================================================

def run_day37_verification():
    print("=" * 70)
    print("🧪 Day 37: Policy Gate Multi-Tool Graph Verification")
    print("=" * 70)

    # 初始化状态
    initial_state: AgentState = {
        "messages": [],
        "execution_plan": [],
        "balance": None,
        "debit_count": 0,
        "approval_given": False  # 未获得人工确认
    }

    # 模拟图的顺序流转（等价于 StateGraph 调度）
    state = agent_node(initial_state)
    state = policy_gate_node(state)
    state = safe_execution_node(state)

    print("\n--- 运行结果检查 ---")
    print(f"当前账户余额: {backend.balance}")
    print(f"实际扣款次数 (debit_count): {backend.debit_count}")

    # 严格断言验证
    assert backend.debit_count == 0, f"严重安全漏洞：未授权转账执行了！扣款次数为 {backend.debit_count}"
    assert state["balance"] == 500.0, "余额应当保持不变"
    
    print("\n✅ 断言全部通过！Graph 的执行路径成功拦截了未授权的高危转账，实际扣款次数为 0。")

if __name__ == "__main__":
    run_day37_verification()

2026-10-20 12:00:41,883 - [INFO] - 🤖 [agent_node] LLM 正在生成批量 Tool Calls...
2026-10-20 12:00:41,884 - [INFO] - 🛡️ [policy_gate_node] 正在对所有 Tool Calls 进行独立契约与安全审查...
2026-10-20 12:00:41,885 - [INFO] - ⚡ [safe_execution_node] 严格根据安全执行计划调度工具...
2026-10-20 12:00:41,886 - [INFO] - ⚙️ [Tool Execution] 执行只读工具: get_balance()
2026-10-20 12:00:41,887 - [WARNING] - ⚠️ [Business Guard] 余额检查结果 (500.0) 小于转账金额！自动取消后续待执行转账。


🧪 Day 37: Policy Gate Multi-Tool Graph Verification

--- 运行结果检查 ---
当前账户余额: 500.0
实际扣款次数 (debit_count): 0

✅ 断言全部通过！Graph 的执行路径成功拦截了未授权的高危转账，实际扣款次数为 0。
